In [1]:
import sys
print(sys.executable)   # should end in pharma-rag/.venv/bin/python

/Users/Anneleonore/Documents/githubportfolio/pharma-rag/.venv/bin/python


### 1. Open database and load model

In [3]:
import duckdb
import pandas as pd
from sentence_transformers import SentenceTransformer

con = duckdb.connect("../data/pharma.duckdb", read_only=True)   # read only: the index can't be changed by accident
model = SentenceTransformer("all-MiniLM-L6-v2")                   # same model as for the trials
DIM = 384

print("Trials in the index:", con.execute("SELECT count(*) FROM trials").fetchone()[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Trials in the index: 12642


### 2. Define hybrid search function

In [4]:
def search(question: str | None = None, keyword: str | None = None, k: int = 10,
           phase: str | None = None, status: str | None = None,
           interventional_only: bool = False, drug_only: bool = False,
           min_year: int | None = None, min_similarity: float = 0.35) -> pd.DataFrame:
    """Hybrid search over the trials.
    question: search by meaning (e.g. 'EGFR inhibitor for lung cancer')
    keyword:  exact word in title, interventions, summary or drug terms (e.g. 'pembrolizumab')
    The rest are exact filters. Without a keyword, results below min_similarity are dropped."""
    where, params = [], []

    # Meaning: similarity to the question (only if a question is given)
    if question:
        params.append(model.encode(question, normalize_embeddings=True).tolist())
        sim_sql = f"array_cosine_similarity(embedding, ?::FLOAT[{DIM}])"
    else:
        sim_sql = "NULL::FLOAT"

    # Keyword: the word must literally appear somewhere (case-insensitive)
    if keyword:
        where.append("(title ILIKE ? OR interventions ILIKE ? OR summary ILIKE ? OR drug_terms ILIKE ?)")
        params += [f"%{keyword}%"] * 4

    # Exact filters
    if phase:
        where.append("phase = ?"); params.append(phase)
    if status:
        where.append("status = ?"); params.append(status)
    if interventional_only:
        where.append("study_type = 'INTERVENTIONAL'")
    if drug_only:
        where.append("(intervention_types LIKE '%DRUG%' OR intervention_types LIKE '%BIOLOGICAL%')")
    if min_year:
        where.append("start_year >= ?"); params.append(min_year)

    # Threshold: only for pure meaning search (keyword matches are always relevant)
    threshold = "WHERE similarity >= ?" if (question and not keyword) else ""
    if threshold:
        params.append(min_similarity)

    sql = f"""
        WITH scored AS (
            SELECT trial_id, title, conditions, interventions, phase, status, sponsor,
                   start_year, text, {sim_sql} AS similarity
            FROM trials
            {"WHERE " + " AND ".join(where) if where else ""}
        )
        SELECT *, count(*) OVER () AS total_matches
        FROM scored
        {threshold}
        ORDER BY similarity DESC NULLS LAST, start_year DESC
        LIMIT {k}
    """
    return con.execute(sql, params).df()


def show(results, n: int = 5):
    """Print the top results with the text that was embedded."""
    if results.empty:
        print("No results.")
        return
    print(f"Total matches: {results['total_matches'].iloc[0]:,}\n")
    for i, r in results.head(n).iterrows():
        sim = f"{r.similarity:.3f}" if pd.notna(r.similarity) else "—"
        print(f"#{i + 1}  similarity {sim}  |  {r.trial_id}  |  {r.phase}  |  {r.status}  |  {r.sponsor}")
        print(f"    {r.text[:400]}\n")

### 3. Test

In [5]:
show(search(keyword="pembrolizumab"))                                  # all pembrolizumab trials
show(search(keyword="Keytruda"))                                       # brand name
show(search(question="EGFR inhibitor for lung cancer"))                # concept
show(search(question="Alzheimer's disease", status="RECRUITING",
            interventional_only=True, drug_only=True))                 # concept + filters

Total matches: 345

#1  similarity —  |  NCT07751042  |  Phase 1/2  |  RECRUITING  |  Strand Therapeutics Inc.
    Phase 1/2 Study of Intravenous Injection of STX-003 in Advanced Solid Tumors as Monotherapy or in Combination With Pembrolizumab. Conditions: Advanced Solid Tumor (Carcinoma, Non-Small-Cell Lung, Melanoma). Interventions: STX-003, KEYTRUDA®. Protocol Title: Phase 1/2, Open-label, Multi-center, First-in-human Study of the Safety, Tolerability, Pharmacokinetics, Pharmacodynamics and Anti-tumor Activ

#2  similarity —  |  NCT06999980  |  Phase 2  |  RECRUITING  |  Melanoma Institute Australia
    Neo IRENIE (NEOadjuvant Ipilimumab, RElatlimab, NIvolumab Evaluation). Conditions: Cutaneous Melanoma, Mucosal Melanoma (Melanoma). Interventions: Ipilimumab 3mg/kg and nivolumab 1mg/kg, Ipilimumab 1mg/kg Nivolumab 480mg and relatlimab 160mg, Nivolumab 480mg and relatlimab 160mg, Ipilimumab 1mg/kg and nivolumab 3mg/kg, Pembrolizumab 200 mg. This clinical trial is for patients with st